In [1]:
import cv2
import numpy as np
import os
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess_input
from sklearn.decomposition import PCA
import pickle
from tensorflow.keras.models import load_model
import subprocess

In [ ]:
with open('/kaggle/input/detect-mc/detect MC/scaler.pkl', 'rb') as f:
    scaler_2 = pickle.load(f)

# Load the PCA
with open('/kaggle/input/detect-mc/detect MC/pca.pkl', 'rb') as f:
    pca_2 = pickle.load(f)

# Load the SVM classifier
with open('/kaggle/input/detect-mc/detect MC/svm_classifier.pkl', 'rb') as f:
    clf_2 = pickle.load(f)

In [3]:
extractor_2 = load_model('/kaggle/input/detect-mc/detect MC/densenet121_feature_extractor.h5')

In [4]:
def reduce_dimensionality(features, n_components=0.90, fit_pca=None):
    if fit_pca is None:
        pca = PCA(n_components=n_components)
        reduced_features = pca.fit_transform(features)
        return reduced_features, pca
    else:
        reduced_features = fit_pca.transform(features)
        return reduced_features
class_labels = ['yes', 'no']

In [5]:
def IsMCFrame(img):
    """
    Function to check if a frame is an MC frame.
    Args:
    - img: A single video frame (numpy array)

    Returns:
    - 'yes' if the frame is an MC frame, 'no' otherwise.
    """
    # Resize and preprocess the frame for the DenseNet model
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = densenet_preprocess_input(img)

    # Extract features using the preloaded DenseNet model
    features = extractor_2.predict(img[np.newaxis, ...])

    # Scale the features
    features = scaler_2.transform(features)

    # Reduce the dimensionality of the features
    features_reduced = reduce_dimensionality(features, fit_pca=pca_2)

    # Predict using the preloaded SVM classifier
    label = clf_2.predict(features_reduced)

    return class_labels[label[0]]

In [6]:
def cut_subvideo(video_path, start_time, end_time, output_path):
    """Uses FFmpeg to cut the subvideo from start_time to end_time."""
    command = [
        'ffmpeg', '-y', '-i', video_path, '-ss', str(start_time), '-to', str(end_time),
        '-c', 'copy', output_path
    ]
    subprocess.run(command)

def SegmentVideo(video_path, output_dir, sampling=1, waiting=20):
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    video_id = video_path.split('/')[-1].split('.')[0]
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(sampling * fps)  # Convert sampling seconds to frames
    waiting_frames = int(waiting * fps)   # Convert waiting seconds to frames
    frame_count = 0
    flag = 0
    segment_count = 0

    # Presentation time tracking
    start_time = 0
    last_pts = 0
    current_pts = 0

    # Ensure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break  # End of video

        # Calculate current PTS (Presentation Time Stamp)
        current_pts = frame_count / fps

        #if frame_count % frame_interval == 0:
            # Check if this frame is an MC frame
        if IsMCFrame(frame) == 'yes':
            if flag == 0:
                    # First MC frame detected, move forward by waiting seconds
                flag = 1
                    #start_time = current_pts
                frame_count += waiting_frames
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count)
            else:
                    # Stop the current subvideo, cut from start_time to this PTS
                flag = 0
                #segment_count += 1
                output_segment_path = os.path.join(output_dir, f'{video_id}_{start_time}_{current_pts}.mp4')

                    # Use FFmpeg to cut subvideo from start_time to current_pts
                cut_subvideo(video_path, start_time, current_pts, output_segment_path)
                print('start time: ', start_time)
                print('current_pts: ', current_pts)
                    # Update start_time for the next segment
                start_time = current_pts
                #frame_count += waiting_frames  # Skip ahead by 'waiting' seconds
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count)
        else:
                # Move forward by sampling seconds
            frame_count += frame_interval
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count)
        #frame_count += 1

    cap.release()
    print(f"Video segmented into {segment_count} segments.")

In [ ]:
folder_path = '/kaggle/input/aic2024-round1-data/video'
output_dir = '/kaggle/working/'
for filename in os.listdir(folder_path):
        if filename.startswith('L18') and filename.endswith('.mp4'):
            video_path = os.path.join(folder_path, filename)
            SegmentVideo(video_path, output_dir)